## Linked Lists · Study Notes

KTH  
Aug 12, 2026

---
> The S-expressions are formed according to the following recursive rules.
> 1. The atomic symbols p₁, p₂, etc., are S-expressions.
> 2. A null expression ∧ is also admitted.
> 3. If *e* is an S-expression so is (*e*).
> 4. If *e₁* and *e₂* are S-expressions so is (*e₁*, *e₂*).
>
> — "Recursive Functions Of Symbolic Expressions," J. McCarthy, 1959

These notes cover the chapter's conceptual material: linked list fundamentals,
the `ListNode` prototype, the boot camp's basic list API (search, insert,
delete), the Top Tips table, and the library discussion. Code cells are
runnable.

## 1. Linked list fundamentals

A **list** implements an ordered collection of values, which may include
repetitions. Specifically, a **singly linked list** is a data structure
containing a sequence of nodes, where each node contains an object and a
reference to the next node. The first node is the **head**; the last node is the
**tail**, and the tail's `next` field is null.

**Variants.** In a **doubly linked list**, each node also has a link to its
predecessor. A **sentinel node** or a **self-loop** can be used instead of null
to mark the end of the list.

### List vs. array

A list is similar to an array in that it contains objects in a linear order. The
key differences:

| Operation | Linked list | Array |
|---|---|---|
| Insert / delete an element | `O(1)` | `O(n)` (shifting) |
| Obtain the *k*-th element | `O(n)` — expensive | `O(1)` |

Lists are usually **building blocks of more complex data structures**. However,
as this chapter shows, they can be the subject of tricky problems in their own
right.

**Figure 4.1** — a singly linked list holding ⟨2, 3, 5, 3, 2⟩, where each node
sits at its own memory address (0x1354, 0x1200, 0x2200, 0x1000, 0x2110) — the
nodes need not be contiguous, unlike an array.

**Figure 4.2** — a doubly linked list holding the same values, with links in
both directions.

## 2. The `ListNode` prototype

For all problems in this chapter, unless otherwise stated, each node has two
entries — a **data** field and a **next** field pointing to the next node, with
the `next` field of the last node being null.

In [1]:
class ListNode:
    def __init__(self, data=0, next_node=None):
        self.data = data
        self.next = next_node

In [2]:
# Helpers (not from EPI) so the boot camp code below is runnable and inspectable
def build_list(values):
    '''Build a singly linked list from a Python list; return the head.'''
    head = None
    for v in reversed(values):
        head = ListNode(v, head)
    return head

def to_python_list(head):
    '''Walk the list and collect the data fields.'''
    out = []
    while head:
        out.append(head.data)
        head = head.next
    return out

L = build_list([2, 3, 5, 3, 2])      # the list from Figure 4.1
print(to_python_list(L))

[2, 3, 5, 3, 2]


## 3. Linked lists boot camp

There are **two types** of list-related problems:
1. Those where you have to **implement your own list**.
2. Those where you have to **exploit the standard list library**.

Implementing a basic list API — **search, insert, delete** — for singly linked
lists is an excellent way to become comfortable with lists.

### 3.1 Search for a key

In [3]:
def search_list(L, key):
    while L and L.data != key:
        L = L.next
    # If key was not present in the list, L will have become null.
    return L

In [4]:
L = build_list([2, 3, 5, 3, 2])

found = search_list(L, 5)
print("search for 5 ->", to_python_list(found))    # returns the node AND its tail

missing = search_list(L, 42)
print("search for 42 ->", missing)                 # None — key absent

search for 5 -> [5, 3, 2]
search for 42 -> None


### 3.2 Insert a new node after a specified node

In [5]:
# Insert new_node after node.
def insert_after(node, new_node):
    new_node.next = node.next
    node.next = new_node

In [6]:
L = build_list([2, 3, 5])
insert_after(L, ListNode(99))        # insert 99 after the head
print(to_python_list(L))             # [2, 99, 3, 5]

[2, 99, 3, 5]


**Order matters.** `new_node.next = node.next` must come *first*. Reversing the
two lines would overwrite `node.next` before it's been read, orphaning the rest
of the list.

### 3.3 Delete a node

In [7]:
# Delete the node past this one. Assume node is not a tail.
def delete_after(node):
    node.next = node.next.next

In [8]:
L = build_list([2, 3, 5, 7])
delete_after(L)                      # delete the node after the head (3)
print(to_python_list(L))             # [2, 5, 7]

[2, 5, 7]


**Complexity.** Insert and delete are **local operations** with `O(1)` time
complexity. Search requires traversing the entire list — e.g., if the key is at
the last node or is absent — so its time complexity is `O(n)`, where `n` is the
number of nodes.

> ⚠️ Note the precondition on `delete_after`: **the node must not be a tail.**
> If `node.next` is `None`, `node.next.next` raises an `AttributeError`. The
> book states the assumption rather than guarding it — in your own code, decide
> deliberately whether to guard or document.

In [9]:
# Demonstrating the precondition
tail = build_list([1])       # single node; tail.next is None
try:
    delete_after(tail)
except AttributeError as e:
    print("delete_after on a tail ->", type(e).__name__, ":", e)

delete_after on a tail -> AttributeError : 'NoneType' object has no attribute 'next'


## 4. Table 4.1 — Top Tips for Linked Lists

- List problems often have a simple **brute-force solution using `O(n)` space**,
  but a subtler solution that **uses the existing list nodes** to reduce space
  complexity to `O(1)`.
- Very often, a problem on lists is **conceptually simple** — it's more about
  cleanly coding what's specified than about designing an algorithm.
- Consider using a **dummy head** (sometimes called a **sentinel**) to avoid
  having to check for empty lists. This simplifies code and makes bugs less
  likely.
- It's easy to **forget to update `next`** (and `previous`, for a doubly linked
  list) for the **head and tail**.
- Algorithms operating on singly linked lists often benefit from using **two
  iterators** — one ahead of the other, or one advancing quicker than the other.

### 4.1 The dummy-head idiom in practice

Illustrating the third tip: without a sentinel, inserting at the front requires
a special case for "list is empty" or "insert before head." With one, every
insertion is uniform.

In [10]:
def insert_sorted(head, value):
    '''Insert value into a sorted list, using a dummy head to avoid special cases.'''
    dummy = ListNode(0, head)        # sentinel sits before the real head
    prev = dummy
    while prev.next and prev.next.data < value:
        prev = prev.next
    insert_after(prev, ListNode(value))
    return dummy.next                # the real head (possibly the new node)

L = None
for v in [5, 2, 8, 1]:
    L = insert_sorted(L, v)          # works even when L is empty — no special case
print(to_python_list(L))

[1, 2, 5, 8]


### 4.2 The two-iterator idiom

Illustrating the fifth tip: advancing one iterator `k` steps ahead of another
finds the `k`-th node from the end in a single pass.

In [11]:
def kth_from_end(head, k):
    fast = head
    for _ in range(k):
        if not fast:
            return None
        fast = fast.next
    slow = head
    while fast:                      # fast stays exactly k nodes ahead
        fast, slow = fast.next, slow.next
    return slow

L = build_list([1, 2, 3, 4, 5])
print("2nd from end:", kth_from_end(L, 2).data)   # 4

2nd from end: 4


## 5. Know your linked list libraries

A reminder up front: **many interview problems directly concerned with lists
require you to write your own list class.**

Under the hood, the Python **`list`** type is typically implemented as a
**dynamically resized array** — not a linked list. (Its key methods are covered
in the Arrays chapter.) This chapter is concerned specifically with **linked
lists, which are not a standard type in Python**, so we define our own singly
and doubly linked list types.

Key methods such a list type typically includes:
- Returning the **head** / **tail**
- **Adding** an element at the head / tail
- Returning the **value stored** at the head / tail
- **Deleting** the head, the tail, or an arbitrary node in the list

> 📝 **Practical aside.** `collections.deque` is a doubly linked list under the
> hood and gives `O(1)` operations at both ends — useful when you need
> linked-list *behavior* rather than linked-list *node manipulation*. But
> interview problems in this chapter want explicit pointer work, so `deque`
> won't substitute.